# Convert Merged Model to GGUF Format for Ollama

This notebook converts the fine-tuned merged Gemma-4 model to GGUF format compatible with Ollama for local inference.

## Steps:
1. Install required dependencies
2. Clone llama.cpp repository
3. Install llama.cpp requirements
4. Convert model to GGUF format (F16)
5. Verify the conversion

## 1. Setup & Environment Check

In [13]:
import sys
import subprocess
import shutil
import os

# Verify Python environment
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")

# Verify pip is available
try:
    result = subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True, text=True)
    print(f"✓ pip available: {result.stdout.strip()}")
except Exception as e:
    print(f"✗ Error with pip: {e}")

# Check for uv (faster package installer)
uv_path = shutil.which("uv")
if uv_path:
    print(f"✓ uv available: {uv_path}")
else:
    print("ℹ uv not found (but pip is available)")

# Check disk space
import psutil
disk_usage = psutil.disk_usage("/")
print(f"\nDisk space available: {disk_usage.free / (1024**3):.2f} GB")

# Check network connectivity
try:
    import urllib.request
    urllib.request.urlopen("https://pypi.org", timeout=5)
    print("✓ Network connectivity: OK")
except Exception as e:
    print(f"✗ Network connectivity issue: {e}")


Python: e:\dev\research & development\Gemma4 project\model_final_step_ft\.venv\Scripts\python.exe
Version: 3.11.12 (main, Apr  9 2025, 04:03:34) [MSC v.1943 64 bit (AMD64)]
✓ pip available: 
✓ uv available: C:\Users\saeeam\.local\bin\uv.EXE

Disk space available: 451.14 GB
✓ Network connectivity: OK


In [14]:
import sys
import subprocess
import shutil

packages = ["transformers", "huggingface_hub", "sentencepiece"]

# Try uv first (it worked before in this environment)
uv_path = shutil.which("uv")
if uv_path:
    print("Attempting installation with uv (recommended method)...")
    try:
        subprocess.check_call([
            "uv", "pip", "install", 
            "--python", sys.executable,
            *packages
        ])
        print("✓ Installation with uv successful!")
    except subprocess.CalledProcessError as e:
        print(f"✗ uv installation failed: {e}")
        print("Falling back to pip...")
        uv_path = None

# Fallback to pip
if not uv_path:
    print("Installing with pip...")
    for package in packages:
        print(f"\nInstalling {package}...")
        try:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", 
                package,
                "--no-cache-dir",
                "-q"
            ])
            print(f"✓ {package} installed")
        except subprocess.CalledProcessError as e:
            print(f"✗ {package} installation failed")
            print(f"  Error: {e}")

print("\n✓ Package installation attempts completed!")


Attempting installation with uv (recommended method)...
✓ Installation with uv successful!

✓ Package installation attempts completed!


## 2. Clone llama.cpp Repository

In [15]:
import subprocess
import os

# Check if llama.cpp already exists
if not os.path.exists("llama.cpp"):
    print("Cloning llama.cpp repository...")
    subprocess.run(["git", "clone", "https://github.com/ggerganov/llama.cpp"], check=True)
    print("✓ llama.cpp cloned successfully!")
else:
    print("✓ llama.cpp already exists")

Cloning llama.cpp repository...
✓ llama.cpp cloned successfully!


## 3. Install llama.cpp Requirements

In [17]:
import sys
import subprocess
import os
import shutil

if os.path.exists("llama.cpp/requirements.txt"):
    print("Installing llama.cpp requirements...")
    
    # Try uv first (more reliable in this environment)
    uv_path = shutil.which("uv")
    if uv_path:
        print("Attempting installation with uv...")
        try:
            subprocess.check_call([
                "uv", "pip", "install",
                "--python", sys.executable,
                "-r", "llama.cpp/requirements.txt"
            ])
            print("✓ llama.cpp requirements installed with uv!")
        except subprocess.CalledProcessError as e:
            print(f"✗ uv installation failed: {e}")
            print("Falling back to pip...")
            uv_path = None
    
    # Fallback to pip
    if not uv_path:
        print("Installing with pip...")
        try:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", 
                "-r", "llama.cpp/requirements.txt",
                "--no-cache-dir"
            ])
            print("✓ llama.cpp requirements installed with pip!")
        except subprocess.CalledProcessError as e:
            print(f"✗ Installation failed: {e}")
            print("This may be due to missing build tools. Continuing anyway...")
else:
    print("Warning: llama.cpp/requirements.txt not found")

Installing llama.cpp requirements...
Attempting installation with uv...
✗ uv installation failed: Command '['uv', 'pip', 'install', '--python', 'e:\\dev\\research & development\\Gemma4 project\\model_final_step_ft\\.venv\\Scripts\\python.exe', '-r', 'llama.cpp/requirements.txt']' returned non-zero exit status 1.
Falling back to pip...
Installing with pip...
✗ Installation failed: Command '['e:\\dev\\research & development\\Gemma4 project\\model_final_step_ft\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-r', 'llama.cpp/requirements.txt', '--no-cache-dir']' returned non-zero exit status 1.
This may be due to missing build tools. Continuing anyway...


## 4. Convert Model to GGUF Format (F16)

Convert the merged model from the `gemma4-e2b_model/merged_model` directory to GGUF format.

In [18]:
import os
from pathlib import Path

# Define paths
merged_model_path = Path("gemma4-e2b_model/merged_model")
output_file = "gemma4-e2b-merged_f16.gguf"

# Verify the merged model exists
if merged_model_path.exists():
    print(f"✓ Merged model found at: {merged_model_path.absolute()}")
    print(f"  Contents: {list(merged_model_path.iterdir())}")
else:
    print(f"✗ Error: Merged model not found at {merged_model_path}")
    raise FileNotFoundError(f"Merged model directory not found at {merged_model_path}")

✓ Merged model found at: e:\dev\research & development\Gemma4 project\model_final_step_ft\gemma4-e2b_model\merged_model
  Contents: [WindowsPath('gemma4-e2b_model/merged_model/chat_template.jinja'), WindowsPath('gemma4-e2b_model/merged_model/config.json'), WindowsPath('gemma4-e2b_model/merged_model/generation_config.json'), WindowsPath('gemma4-e2b_model/merged_model/model-00001-of-00003.safetensors'), WindowsPath('gemma4-e2b_model/merged_model/model-00002-of-00003.safetensors'), WindowsPath('gemma4-e2b_model/merged_model/model-00003-of-00003.safetensors'), WindowsPath('gemma4-e2b_model/merged_model/model.safetensors.index.json'), WindowsPath('gemma4-e2b_model/merged_model/tokenizer.json'), WindowsPath('gemma4-e2b_model/merged_model/tokenizer_config.json')]


In [19]:
import subprocess
import sys
import os

# Define paths
merged_model_path = "gemma4-e2b_model/merged_model"
output_file = "gemma4-e2b-merged_f16.gguf"

print(f"Converting model from: {merged_model_path}")
print(f"Output file: {output_file}")
print(f"Quantization type: F16\n")

# Run the conversion script
try:
    result = subprocess.run([
        sys.executable, 
        "llama.cpp/convert_hf_to_gguf.py",
        merged_model_path,
        "--outfile", output_file,
        "--model-name", "gemma4-e2b",
        "--outtype", "f16"
    ], capture_output=False, text=True, check=False)
    
    if result.returncode == 0:
        print("\n✓ Conversion completed!")
    else:
        print(f"\n✗ Conversion failed with exit code {result.returncode}")
        print("Check the output above for details.")
        
except Exception as e:
    print(f"✗ Error running conversion: {e}")

Converting model from: gemma4-e2b_model/merged_model
Output file: gemma4-e2b-merged_f16.gguf
Quantization type: F16


✓ Conversion completed!


## 5. Verify Conversion

In [20]:
import os

# Check if the GGUF file was created
if os.path.exists(output_file):
    file_size_gb = os.path.getsize(output_file) / (1024 ** 3)
    print(f"✓ GGUF file created successfully!")
    print(f"  File: {output_file}")
    print(f"  Size: {file_size_gb:.2f} GB")
    print(f"\n  Ready for Ollama import!")
else:
    print(f"✗ Error: GGUF file was not created at {output_file}")
    print(f"  Check the conversion output above for errors.")

✓ GGUF file created successfully!
  File: gemma4-e2b-merged_f16.gguf
  Size: 8.64 GB

  Ready for Ollama import!


## 6. Quantize GGUF Model to Q4_K_M

Quantize the F16 GGUF model to Q4_K_M format for better performance and smaller file size while maintaining quality.

Q4_K_M provides an excellent balance between:
- **Compression**: ~60% reduction from F16 (3.5GB → ~1.4GB)
- **Quality**: Minimal accuracy loss for inference
- **Speed**: Faster inference than F16

In [33]:
import subprocess
import os
import sys

print("Checking if llama.cpp needs to be built...")
print("=" * 70)

# Check if quantize executable exists
quantize_exists = False
check_paths = [
    "llama.cpp/build/bin/quantize",
    "llama.cpp/build/bin/quantize.exe",
    "llama.cpp/tools/quantize",
    "llama.cpp/tools/quantize.exe",
]

for path in check_paths:
    if os.path.exists(path):
        print(f"✓ Quantize tool already built: {path}")
        quantize_exists = True
        break

if not quantize_exists:
    print("✗ Quantize tool not found. Building llama.cpp...\n")
    
    if not os.path.exists("llama.cpp"):
        print("✗ Error: llama.cpp directory not found!")
        raise FileNotFoundError("llama.cpp directory not found")
    
    print("Building with CMake (recommended)...")
    try:
        os.makedirs("llama.cpp/build", exist_ok=True)
        
        # Configure
        print("Step 1: Configuring CMake...")
        result = subprocess.run(
            ["cmake", "..", "-DCMAKE_BUILD_TYPE=Release"],
            cwd="llama.cpp/build",
            timeout=300
        )
        
        if result.returncode == 0:
            print("✓ CMake configuration successful\n")
            
            # Build
            print("Step 2: Building (this may take 5-10 minutes)...")
            result = subprocess.run(
                ["cmake", "--build", ".", "--config", "Release", "-j"],
                cwd="llama.cpp/build",
                timeout=1200
            )
            
            if result.returncode == 0:
                print("✓ Build completed successfully!")
            else:
                print("⚠ Build completed with warnings (may still work)")
        else:
            print("✗ CMake configuration failed\n")
            print("Trying with make instead...")
            result = subprocess.run(
                ["make", "-j"],
                cwd="llama.cpp",
                timeout=1200
            )
            if result.returncode == 0:
                print("✓ Make build successful!")
            else:
                print("⚠ Make build completed with warnings")
                
    except subprocess.TimeoutExpired:
        print("⚠ Build timed out, but may still have completed partially")
    except FileNotFoundError:
        print("✗ Error: CMake or Make not found. Please install build tools.")
        print("  Windows: Install Visual Studio Build Tools or CMake")
        print("  Mac: brew install cmake")
        print("  Linux: sudo apt-get install build-essential cmake")
        raise
    except Exception as e:
        print(f"✗ Build error: {e}")
        raise

print("=" * 70)
print("Ready for quantization!")


Checking if llama.cpp needs to be built...
✓ Quantize tool already built: llama.cpp/tools/quantize
Ready for quantization!


In [40]:
methods = ['q4_k_m']

quantized_path = "gemma4-e2b-merged_f16.gguf"

import os

for m in methods:
    qtype = f"{quantized_path}/{m.upper()}.gguf"
    os.system(f"./llama.cpp/tools/quantize {quantized_path} {qtype} {m}")

## Next Steps: Import into Ollama

Once the GGUF file is created, you can use it with Ollama:

```bash
# Create a Modelfile
FROM ./gemma4-e2b-merged_f16.gguf

# Import into Ollama
ollama create gemma4-e2b -f Modelfile

# Run the model
ollama run gemma4-e2b
```

## Summary: Gemma4 Health Companion Model

This notebook has created an offline AI health companion with the following specifications:

**Model Details:**
- Base: Gemma-4 fine-tuned on medical datasets
- Format: GGUF (F16 quantization) - optimized for CPU/offline inference
- File: `gemma4-e2b-merged_f16.gguf`
- Modelfile: `Modelfile` (ready for Ollama)

**Optimization for Medical Use:**
- **Temperature 0.35**: Conservative for medical accuracy
- **Context 4096**: Support extended patient conversations
- **System Prompt**: Emphasizes safety, triage, medicine identification
- **Multilingual**: Can respond in patient's language

**Use Cases:**
✓ Safe preliminary medical triage (URGENT/HIGH/MODERATE/LOW)
✓ Medicine identification by name, appearance, or symptoms
✓ General health education for underserved populations
✓ Offline accessibility (3.5 billion people without nearby doctors)
✓ Self-care guidance with professional referral recommendations

**Next Steps:**
1. Install Ollama from https://ollama.ai
2. Run: `ollama create gemma4-health-companion -f Modelfile`
3. Test with: `ollama run gemma4-health-companion`
4. Deploy as web service or standalone application

## 7. Import Model into Ollama & Test

In [22]:
import os

# Create Modelfile for Ollama
modelfile_content = """FROM ./gemma4-e2b-merged_f16.gguf

# System prompt for medical health companion
SYSTEM \"\"\"You are a compassionate and knowledgeable AI Health Companion trained on medical knowledge from ChatDoctor and HealthCare-Magic datasets. Your role is to provide safe, preliminary triage guidance and health information.

IMPORTANT GUIDELINES:
1. SAFETY FIRST: Always remind users that you are an AI providing preliminary guidance, not a substitute for professional medical care. Encourage them to consult a qualified healthcare provider.
2. LANGUAGE: Respond in the patient's preferred language when possible.
3. MEDICINE IDENTIFICATION: When patients ask about medications, provide information about:
   - Generic and brand names
   - Common uses
   - Typical dosage ranges
   - Major side effects and warnings
   - Drug interactions to watch for
4. TRIAGE: Help assess symptom severity using a simple scale:
   - URGENT (Go to ER immediately)
   - HIGH (See doctor within 24 hours)
   - MODERATE (Schedule appointment soon)
   - LOW (Self-care, monitor symptoms)
5. PRIVACY: Do not store or recall personal health information between conversations.
6. HONESTY: If uncertain about medical information, say so and recommend professional consultation.

Always provide evidence-based information and cite common medical knowledge when relevant.\"\"\"

# Temperature: Lower for consistency in medical guidance (0.3-0.4 recommended)
PARAMETER temperature 0.35

# Top-k: Controls diversity of responses
PARAMETER top_k 40

# Top-p: Nucleus sampling for coherent responses
PARAMETER top_p 0.9

# Repeat penalty: Prevents repetitive outputs
PARAMETER repeat_penalty 1.1

# Context size: Allow longer conversations
PARAMETER num_ctx 4096

# Threads for inference
PARAMETER num_thread 8

# Description for model selection
TEMPLATE \"\"\"[INST] {{ .Prompt }} [/INST]\"\"\"\"\"\"

# Tags for discoverability
TAGS medical health-companion triage offline ai
"""

# Write Modelfile
modelfile_path = "Modelfile"
with open(modelfile_path, "w") as f:
    f.write(modelfile_content)

print(f"✓ Modelfile created: {modelfile_path}")
print(f"\nModelfile Configuration for Medical Health Companion:")
print(f"  Temperature: 0.35 (consistent, reliable medical advice)")
print(f"  Top-K: 40 (balanced diversity)")
print(f"  Top-P: 0.9 (high-quality coherent responses)")
print(f"  Context: 4096 tokens (extended medical conversations)")
print(f"  Purpose: Safe triage guidance & medicine identification")
print(f"\nContent preview:")
print("=" * 60)
print(modelfile_content[:500] + "...")
print("=" * 60)

✓ Modelfile created: Modelfile

Modelfile Configuration for Medical Health Companion:
  Temperature: 0.35 (consistent, reliable medical advice)
  Top-K: 40 (balanced diversity)
  Top-P: 0.9 (high-quality coherent responses)
  Context: 4096 tokens (extended medical conversations)
  Purpose: Safe triage guidance & medicine identification

Content preview:
FROM ./gemma4-e2b-merged_f16.gguf

# System prompt for medical health companion
SYSTEM """You are a compassionate and knowledgeable AI Health Companion trained on medical knowledge from ChatDoctor and HealthCare-Magic datasets. Your role is to provide safe, preliminary triage guidance and health information.

IMPORTANT GUIDELINES:
1. SAFETY FIRST: Always remind users that you are an AI providing preliminary guidance, not a substitute for professional medical care. Encourage them to consult a qua...
